# EDA — Tasas de interés activas por tipo de crédito (2025)
**Fuente:** [datos.gov.co — Superfinanciera](https://www.datos.gov.co/Econom-a-y-Finanzas/Tasas-de-inter-s-activas-por-tipo-de-cr-dito-Hist-/w9zh-vetq/about_data)
**Dataset ID (Socrata):** `w9zh-vetq`
**Registros totales en 2025 (confirmado por conteo server-side):** 36,820,700
**Granularidad temporal real:** semanal — 52 `fecha_corte`, todas viernes, del 2025-01-03 al 2025-12-26.

## ¿Qué es `fecha_corte`?
Es el **viernes de cierre de la semana de reporte**. Cada fila resume (agrupa) los créditos que una entidad desembolsó *durante esa semana*, con la misma combinación exacta de características (tipo de crédito, producto, plazo, perfil del deudor, etc.). No es la fecha de un desembolso individual.

## Estrategia de descarga
Con 36.8M de filas, un solo `$offset` gigante es lento e inestable en Socrata. En cambio, aprovechamos que ya conocemos las 52 fechas de corte exactas y **descargamos semana por semana, en paralelo** (varios hilos), lo cual es mucho más rápido y confiable.

Puedes controlar cuánto descargar con `SEMANAS_A_DESCARGAR` más abajo:
- `None` → las 52 semanas completas (2025 entero, ~36.8M filas — tardará bastante).
- Un número, ej. `4` → solo las primeras N semanas (para explorar rápido, ej. 4 semanas ≈ 2.8M filas).
- Una lista de fechas específicas, ej. `["2025-01-03", "2025-06-06"]` → semanas puntuales.

> 💡 Si tienes un *app token* de datos.gov.co (`https://www.datos.gov.co/profile/app_tokens`), ponlo en `APP_TOKEN` — acelera la descarga y evita bloqueos por límite de tasa.


In [ ]:
import time
import concurrent.futures as cf

import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pyarrow as pa
import pyarrow.parquet as pq

pd.set_option("display.max_columns", 100)
plt.rcParams["figure.figsize"] = (10, 5)

DATASET_ID = "w9zh-vetq"
BASE_URL   = f"https://www.datos.gov.co/resource/{DATASET_ID}.json"
APP_TOKEN  = None  # <- opcional: tu token de https://www.datos.gov.co/profile/app_tokens
HEADERS    = {"X-App-Token": APP_TOKEN} if APP_TOKEN else {}

# Nombres de columna reales (confirmados)
COL_FECHA   = "fecha_corte"
COL_TASA    = "tasa_efectiva_promedio"
COL_MARGEN  = "margen_adicional_a_la"
COL_TIPO    = "tipo_de_cr_dito"
COL_PRODUCTO = "producto_de_cr_dito"
COL_PLAZO   = "plazo_de_cr_dito"
COL_GARANTIA = "tipo_de_garant_a"
COL_ENTIDAD = "nombre_entidad"
COL_TIPO_ENTIDAD = "nombre_tipo_entidad"
COL_PERSONA = "tipo_de_persona"
COL_SEXO    = "sexo"
COL_TAM_EMPRESA = "tama_o_de_empresa"
COL_ANTIGUEDAD = "antiguedad_de_la_empresa"
COL_ETNIA   = "grupo_etnico"
COL_TASA_REF = "tipo_de_tasa"
COL_RANGO_MONTO = "rango_monto_desembolsado"
COL_CLASE_DEUDOR = "clase_deudor"
COL_MONTO   = "montos_desembolsados"
COL_NCREDITOS = "numero_de_creditos"
COL_CIIU    = "codigo_ciiu"
COL_MUNICIPIO = "codigo_municipio"


## 1. Confirmar las 52 fechas de corte de 2025
(Ya lo validamos antes: 52 valores, todos viernes. Lo repetimos aquí para que el cuaderno sea autocontenido y reproducible.)

In [ ]:
where_2025 = f"{COL_FECHA} between '2025-01-01T00:00:00' and '2025-12-31T23:59:59'"

resp = requests.get(
    BASE_URL,
    headers=HEADERS,
    params={"$select": COL_FECHA, "$where": where_2025, "$group": COL_FECHA, "$order": COL_FECHA, "$limit": 1000},
    timeout=30,
).json()

fechas_2025 = sorted(pd.to_datetime([r[COL_FECHA] for r in resp]).strftime("%Y-%m-%d").tolist())
print(f"Semanas encontradas en 2025: {len(fechas_2025)}")
fechas_2025[:5], fechas_2025[-5:]


## 2. Elegir cuántas semanas descargar
Ajusta esto según qué tan rápido necesitas iterar. Para una primera exploración recomiendo empezar con pocas semanas y luego ampliar.

In [ ]:
SEMANAS_A_DESCARGAR = 4   # <-- AJUSTAR: None = 2025 completo (36.8M filas), o un número, o una lista de fechas

if SEMANAS_A_DESCARGAR is None:
    semanas = fechas_2025
elif isinstance(SEMANAS_A_DESCARGAR, int):
    semanas = fechas_2025[:SEMANAS_A_DESCARGAR]
else:
    semanas = list(SEMANAS_A_DESCARGAR)

print(f"Se descargarán {len(semanas)} semana(s): {semanas}")


## 3. Descarga paralela por semana
Cada semana se pagina con `$limit`/`$offset` (offset acotado dentro de la semana, no sobre las 36.8M filas totales), y varias semanas se descargan **al mismo tiempo** con un `ThreadPoolExecutor`. Esto es mucho más rápido que una descarga secuencial de un solo offset gigante.

In [ ]:
PAGE_SIZE = 50_000
MAX_WORKERS = 6   # <-- AJUSTAR según tu conexión / límites de la API (sube si tienes APP_TOKEN)
OUT_FILE = "tasas_2025.parquet"

def descargar_semana(fecha_corte_str, session, page_size=PAGE_SIZE):
    frames = []
    offset = 0
    where_semana = f"{COL_FECHA} = '{fecha_corte_str}T00:00:00.000'"
    while True:
        params = {"$where": where_semana, "$limit": page_size, "$offset": offset, "$order": ":id"}
        r = session.get(BASE_URL, headers=HEADERS, params=params, timeout=60)
        r.raise_for_status()
        lote = r.json()
        if not lote:
            break
        frames.append(pd.DataFrame(lote))
        offset += page_size
        if len(lote) < page_size:
            break
    df_semana = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    return fecha_corte_str, df_semana

t0 = time.time()
resultados = {}
with requests.Session() as session:
    with cf.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futuros = {executor.submit(descargar_semana, f, session): f for f in semanas}
        for futuro in cf.as_completed(futuros):
            fecha, df_semana = futuro.result()
            resultados[fecha] = df_semana
            print(f"  Semana {fecha}: {len(df_semana):,} filas")

df = pd.concat([resultados[f] for f in semanas], ignore_index=True)
print(f"\nTotal descargado: {len(df):,} filas en {time.time() - t0:.1f} s")

df.to_parquet(OUT_FILE, index=False)
print(f"Guardado en {OUT_FILE}")


## 4. Cargar y tipar el subconjunto
A partir de aquí trabajamos sobre el `.parquet` local, no sobre la API.

In [ ]:
df = pd.read_parquet(OUT_FILE)

df[COL_FECHA] = pd.to_datetime(df[COL_FECHA])
for c in [COL_TASA, COL_MARGEN, COL_MONTO, COL_NCREDITOS]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df["semana"] = df[COL_FECHA].dt.isocalendar().week
df["mes"] = df[COL_FECHA].dt.month

print(df.shape)
df.head()


## 5. Calidad de datos: nulos y duplicados

In [ ]:
nulos = df.isna().sum().sort_values(ascending=False)
pct_nulos = (nulos / len(df) * 100).round(2)
resumen_nulos = pd.DataFrame({"n_nulos": nulos, "pct_nulos": pct_nulos})
resumen_nulos[resumen_nulos["n_nulos"] > 0]


In [ ]:
print(f"Filas duplicadas: {df.duplicated().sum():,}")


## 6. Estadísticas descriptivas de la tasa efectiva promedio

In [ ]:
df[COL_TASA].describe()


In [ ]:
df[COL_TASA].plot(kind="hist", bins=60, title="Distribución de la tasa efectiva promedio")
plt.xlabel(COL_TASA)
plt.show()


## 7. Tasa promedio por tipo de crédito

In [ ]:
resumen_tipo = (
    df.groupby(COL_TIPO)[COL_TASA]
      .agg(["count", "mean", "median", "std"])
      .sort_values("mean", ascending=False)
)
resumen_tipo


In [ ]:
resumen_tipo["mean"].sort_values().plot(kind="barh", title="Tasa promedio por tipo de crédito")
plt.xlabel(COL_TASA)
plt.tight_layout()
plt.show()


## 8. Enfoque diferencial: tasa por sexo y tamaño de empresa
Este dataset permite mirar brechas de acceso al crédito. Dos cortes rápidos:

In [ ]:
df.groupby(COL_SEXO)[COL_TASA].agg(["count", "mean", "median"])


In [ ]:
df.groupby(COL_TAM_EMPRESA)[COL_TASA].agg(["count", "mean", "median"]).sort_values("mean")


## 9. Top entidades por número de créditos y tasa promedio

In [ ]:
top_entidades = (
    df.groupby(COL_ENTIDAD)
      .agg(n_creditos=(COL_NCREDITOS, "sum"), monto_total=(COL_MONTO, "sum"), tasa_prom=(COL_TASA, "mean"))
      .sort_values("n_creditos", ascending=False)
      .head(20)
)
top_entidades


## 10. Serie de tiempo semanal

In [ ]:
serie_semanal = df.groupby(COL_FECHA)[COL_TASA].mean()
serie_semanal.plot(marker="o", title="Tasa efectiva promedio por semana")
plt.xlabel("Fecha de corte")
plt.ylabel(COL_TASA)
plt.grid(alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


### Por tipo de crédito

In [ ]:
pivote = df.pivot_table(index=COL_FECHA, columns=COL_TIPO, values=COL_TASA, aggfunc="mean")
pivote.plot(marker="o", title="Tasa promedio semanal por tipo de crédito")
plt.xlabel("Fecha de corte")
plt.ylabel(COL_TASA)
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 11. Outliers (regla IQR) sobre la tasa

In [ ]:
q1, q3 = df[COL_TASA].quantile([0.25, 0.75])
iqr = q3 - q1
lim_inf, lim_sup = q1 - 1.5 * iqr, q3 + 1.5 * iqr

outliers = df[(df[COL_TASA] < lim_inf) | (df[COL_TASA] > lim_sup)]
print(f"Límites: [{lim_inf:.2f}, {lim_sup:.2f}] | Outliers: {len(outliers):,} ({len(outliers)/len(df)*100:.2f}%)")
outliers[[COL_FECHA, COL_ENTIDAD, COL_TIPO, COL_TASA, COL_MONTO]].sort_values(COL_TASA, ascending=False).head(15)


## 12. Amplíar a 2025 completo
Cuando quieras el año entero: sube `SEMANAS_A_DESCARGAR = None` en el paso 2, sube `MAX_WORKERS` si tienes un `APP_TOKEN`, y vuelve a correr los pasos 3 en adelante. Con 52 semanas y descarga en paralelo, es mucho más viable que un solo offset gigante — aun así, calcula tiempo y espacio en disco (36.8M filas en parquet comprimido probablemente sean varios GB).


## 13. Próximo paso: dashboard
Con este `.parquet` ya filtrado, el flujo natural para un dashboard interactivo en Python es:

- **Streamlit** — el más simple: convierte celdas de este notebook en widgets (`st.selectbox`, `st.slider`) y gráficos (`st.plotly_chart`) en pocas líneas, corre con `streamlit run app.py`.
- **Dash (Plotly)** — más control sobre layout/callbacks, curva de aprendizaje algo mayor.
- **Panel / Voila** — alternativas si quieres mantener el notebook casi intacto y exponerlo como app.

Para volumen (millones de filas), no cargues el `.parquet` crudo en el dashboard: pre-agrega (por semana, tipo de crédito, entidad, etc.) igual que en las celdas de arriba, y that agregado (mucho más liviano) es lo que alimenta los gráficos interactivos.
